### Imports, loading data and methods from features.py

In [2]:
from features import *
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import torch.optim as optim
import wandb


data = pd.read_csv('train.csv')

cryosleep(data)
cabin(data)
passenger(data)
vip(data)

X = data.drop(columns=['PassengerId', 'Num', 'Name', 'Transported', 'Cabin'])
y = data['Transported']



X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 42)
X_train, medians = money_columns(X_train)
X_test, _ = money_columns(X_test, medians)

### Categorical and Numerical Features

In [3]:
cat_col = X.select_dtypes(include=['object', 'str']).columns
num_col = X.select_dtypes(include=['number']).columns
print(cat_col)

Index(['HomePlanet', 'Destination', 'Deck', 'Side'], dtype='str')


### Categorical pipeline + OneHotEncoder, SimpleImputer, StandardScaler

In [4]:
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

age_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

### ColumnTransformer

In [18]:
preprocessor = ColumnTransformer([
    ('cat', cat_pipeline, cat_col),
    ('age', age_pipeline, ['Age']),
    ('vip', SimpleImputer(strategy='most_frequent'), ['VIP']),
    ('money', StandardScaler(), ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'MoneySpend']),
], remainder='passthrough')

X_train_preproc = preprocessor.fit_transform(X_train)
X_test_preproc = preprocessor.transform(X_test)

X_train_tensor = torch.tensor(X_train_preproc, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)

X_test_tensor = torch.tensor(X_test_preproc, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

y_train_tensor.shape

torch.Size([6085, 1])

### DataLoaders

In [22]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=False)

torch.Size([6085, 26])

### MLP

In [ ]:
input_dim = X_train_tensor.shape[1]

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.neural_net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.LeakyReLU(),
            nn.Linear(64, 64),
            nn.LeakyReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        result = self.neural_net(x)
        return result

### Training Loop

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def train_model(lr):
    wandb.init(
        project="Spaceship",
        name=f'lr: {lr}',
        config={'lr': lr, 'epochs': 20, 'batch_size': 64}
    )

    model = Model().to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr)

    for epoch in range(20):
        model.train()
        running_loss = 0.0
        for X, y in train_dataloader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            result = model(X)
            loss = criterion(result, y)
            running_loss += loss.item()
            loss.backward()
            optimizer.step()

        avg_loss = running_loss / len(train_dataloader)

        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for X, y in test_dataloader:
                X, y = X.to(device), y.to(device)
                result = model(X)
                prediction = (result > 0).int()
                correct += (prediction == y).sum().item()
                total += len(y)
        accuracy = correct / total

        wandb.log({'loss': avg_loss, 'Accuracy': accuracy, 'epoch': epoch})
    wandb.finish()

for lr in [0.1, 0.01, 0.001]:
    train_model(lr)